In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Monitorear o cancelar un job

Consulta la lista de tus cargas de trabajo en la [página Workloads](https://quantum.cloud.ibm.com/workloads).

## Ver el estado de un job
Ve a tu [tabla Workloads](https://quantum.cloud.ibm.com/workloads) y revisa la columna Status para saber si un job se completó o falló.

## Ver el uso restante
Ve a tu [tabla Instances](https://quantum.cloud.ibm.com/instances) y selecciona la pestaña asociada al plan del que deseas ver el uso restante. Se muestra el tiempo total utilizado y el tiempo total restante en tu plan.

## Ver métricas sobre el número de jobs y cargas de trabajo enviados
Ve a la [página Analytics](https://quantum.cloud.ibm.com/analytics) para ver el número total de jobs enviados, así como el recuento de cargas de trabajo en lote y en sesión. Ten en cuenta que solo puedes ver la página Analytics para las cuentas que posees o administras.

## Monitorear un job
Usa la instancia del job para verificar su estado o recuperar los resultados llamando al comando apropiado:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Revisa los resultados del job inmediatamente después de que se complete. Los resultados están disponibles una vez que el job finaliza. Por lo tanto, job.result() es una llamada bloqueante hasta que el job se complete.                               |
| job.job\_id()                 | Devuelve el ID que identifica de forma única ese job. Recuperar los resultados del job en un momento posterior requiere el ID del job. Por ello, se recomienda guardar los IDs de los jobs que puedas querer recuperar más adelante. |
| job.status()                  | Verifica el estado del job.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | Recupera un job enviado anteriormente. Esta llamada requiere el ID del job.                                                                                                                                      |

<span id="retrieve-later"></span>
## Recuperar resultados de un job en un momento posterior
Llama a `service.job(\<job\_id>)` para recuperar un job que enviaste anteriormente. Si no tienes el ID del job, o si deseas recuperar varios jobs a la vez, incluidos jobs de QPUs (unidades de procesamiento cuántico) retiradas, llama a `service.jobs()` con filtros opcionales. Consulta [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` también devuelve jobs ejecutados desde el paquete obsoleto `qiskit-ibm-provider`. Los jobs enviados por el paquete más antiguo (también obsoleto) `qiskit-ibmq-provider` ya no están disponibles.

### Ejemplo
Este ejemplo devuelve los 10 jobs de runtime más recientes que se ejecutaron en `my_backend`:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>